<a href="https://colab.research.google.com/github/zamoralinokevin-eng/Maestria-en-IA/blob/main/Sesion10_Data_Profiling_Entregable_271753.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo: Kevin Josue Zamora Lino**

**Matrícula: 271753**

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [1]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)

Netflix: (7787, 12)
Marketing: (2240, 29)


---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

In [2]:
#se revisan los nombres de las columnas del dataset
df_marketing.columns

#se crea un copia para conservar el datframe origianl
df_marketing_renombrado = df_marketing.copy()

#se colocan en minusculas las columnas de la copia del df
df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.lower()

#se modifican al menos 2 nombres de columnas con rename
df_marketing_renombrado = df_marketing_renombrado.rename(
    columns={"kidhome":"kid_home", "teenhome":"teen_home", "mntwines": "mnt_wines"})
print(df_marketing_renombrado.columns)


Index(['id', 'year_birth', 'education', 'marital_status', 'income', 'kid_home',
       'teen_home', 'dt_customer', 'recency', 'mnt_wines', 'mntfruits',
       'mntmeatproducts', 'mntfishproducts', 'mntsweetproducts',
       'mntgoldprods', 'numdealspurchases', 'numwebpurchases',
       'numcatalogpurchases', 'numstorepurchases', 'numwebvisitsmonth',
       'acceptedcmp3', 'acceptedcmp4', 'acceptedcmp5', 'acceptedcmp1',
       'acceptedcmp2', 'complain', 'z_costcontact', 'z_revenue', 'response'],
      dtype='object')


---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [3]:
#identificamos valores nulos antes de la conversion de tipo de datos en la columna
print("Nulos antes de conversion:", df_netflix["date_added"].isna().sum())

#convertimos los datos a tipo datetime
df_netflix["date_added"] = pd.to_datetime(df_netflix["date_added"], format="mixed", errors="coerce")

#mostramos valores nulos en la columna despues de conversion de datos
print("Nulos despues de conversion:", df_netflix["date_added"].isna().sum())


Nulos antes de conversion: 10
Nulos despues de conversion: 10


---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [4]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

In [10]:
#se cuentan los duiplicados exactos del datframe
print("Duplicados del dataframe", df_marketing_dup.duplicated().sum())
print("Duplicados de ID", df_marketing_dup.duplicated(subset="ID").sum())

#se eliminan los duplicados de la columna ID y se actualiza el dataframe
df_marketing_dup = df_marketing_dup.drop_duplicates(subset="ID", keep="first")

#se verifica que los duplicados ya no estan prsentes
print("Duplicados restantes de ID", df_marketing_dup.duplicated(subset="ID").sum())


Duplicados del dataframe 2
Duplicados de ID 2
Duplicados restantes de ID 0


---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [13]:
#Se muestran cuando valores nulos en total hay en cada columna
print("Las columnas con valor nulo son\n", df_netflix.isnull().sum())

#se muestran el total de filas que tienen al menos 1 valor nulo en cualquier columna
print("\nEl total de filas con valor nulo son", df_netflix.isnull().any(axis=1).sum())

Las columnas con valor nulo son
 show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64

El total de filas con valor nulo son 2979


---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [14]:
#calculo de la completitud en porcentaje
completitud_porcentaje = round((1 - df_netflix.isnull().sum() / len(df_netflix))*100, 2)
print(completitud_porcentaje)
print("La columna con la completitud mas baja es", completitud_porcentaje.idxmin(), "con", completitud_porcentaje.min(),"%")


show_id         100.00
type            100.00
title           100.00
director         69.32
cast             90.78
country          93.49
date_added       99.87
release_year    100.00
rating           99.91
duration        100.00
listed_in       100.00
description     100.00
dtype: float64
La columna con la completitud mas baja es director con 69.32 %


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

In [15]:
#muestro los diferentes valores registrados de la columna "Marital_Status"
print(df_marketing["Marital_Status"].value_counts())

#Reemplazo Alone dentro de la categoria single y pongo como None los valores YOLO y Absurd
df_marketing["Marital_Status"] = df_marketing["Marital_Status"].replace({"Alone":"Single","YOLO":None,"Absurd":None})

#Elimino los valores nulos
df_marketing.dropna(subset=["Marital_Status"], inplace=True)

#muestro el df con la decision tomada
print(df_marketing["Marital_Status"].value_counts())

print("Decidi reclasificar Alone a single por la similitud en la palabra.")
print("Decidi eliminar Absurd y YOLO porque no guardan valor con el estado marital")

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64
Marital_Status
Married     864
Together    580
Single      483
Divorced    232
Widow        77
Name: count, dtype: int64
Decidi reclasificar Alone a single por la similitud en la palabra.
Decidi eliminar Absurd y YOLO porque no guardan valor con el estado marital


---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [16]:
#se busca el patron en la columna sow_id
cumplimiento = df_netflix["show_id"].str.match(r"^s\d+$").value_counts()

#se realiza el calculo de porcentaje de cumplimiento
cumplimiento_porcentaje = cumplimiento[True] / len(df_netflix) * 100
print("Porcentaje de cumplimiento", cumplimiento_porcentaje, "%")



Porcentaje de cumplimiento 100.0 %


---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

In [20]:
#se utiliza describe() solbre la columna Year_Birth
print(df_marketing[["Year_Birth"]].describe())

print("\nNo tiene sentido el valor minimo ya que se obtuvo en los 1800 y solo hay 1 valor de esa epoca.\n")

#se filtra para mostrar datos menores a 1935
print(df_marketing[df_marketing["Year_Birth"] < 1935])

print("\nA partir de 1935 no se considera error de captura ya que son muchos datos registrados a partir de ese año.")


        Year_Birth
count  2236.000000
mean   1968.796512
std      11.980604
min    1893.000000
25%    1959.000000
50%    1970.000000
75%    1977.000000
max    1996.000000

No tiene sentido el valor minimo ya que se obtuvo en los 1800 y solo hay 1 valor de esa epoca.

        ID  Year_Birth Education Marital_Status   Income  Kidhome  Teenhome  \
192   7829        1900  2n Cycle       Divorced  36640.0        1         0   
239  11004        1893  2n Cycle         Single  60182.0        0         1   
339   1150        1899       PhD       Together  83532.0        0         0   

    Dt_Customer  Recency  MntWines  ...  NumWebVisitsMonth  AcceptedCmp3  \
192  2013-09-26       99        15  ...                  5             0   
239  2014-05-17       23         8  ...                  4             0   
339  2013-09-26       36       755  ...                  1             0   

     AcceptedCmp4  AcceptedCmp5  AcceptedCmp1  AcceptedCmp2  Complain  \
192             0             0      

---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [25]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['Income'] = df_practica['Income'].astype(object)
df_practica.loc[5, 'Income'] = 'sesenta mil'

df_practica[['ID', 'Marital_Status', 'Income']]

,ID,Marital_Status,Income
0,6856,Together,21645.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,6466,Married,57236.0
4,6357,Divorced,59052.0
5,5314,Together,sesenta mil
6,4351,Divorced,37244.0
7,6182,Together,26646.0
8,3790,Together,34633.0
9,9530,Married,24645.0


In [26]:
# Paso 1 — ajuste de tipos
#se revisa el tipo de dato de cada columna del df
print("Tipo de dato antes de la conversion\n", df_practica.dtypes)

#se convierten a numeric la columna Income y datetime la columna Dt_Customer
df_practica["Income"] = pd.to_numeric(df_practica["Income"], errors="coerce")
df_practica["Dt_Customer"] = pd.to_datetime(df_practica["Dt_Customer"], errors="coerce")
print("Tipo de dato despues de la conversion\n", df_practica.dtypes)

Tipo de dato antes de la conversion
 ID                      int64
Year_Birth              int64
Education              object
Marital_Status         object
Income                 object
Kidhome                 int64
Teenhome                int64
Dt_Customer            object
Recency                 int64
MntWines                int64
MntFruits               int64
MntMeatProducts         int64
MntFishProducts         int64
MntSweetProducts        int64
MntGoldProds            int64
NumDealsPurchases       int64
NumWebPurchases         int64
NumCatalogPurchases     int64
NumStorePurchases       int64
NumWebVisitsMonth       int64
AcceptedCmp3            int64
AcceptedCmp4            int64
AcceptedCmp5            int64
AcceptedCmp1            int64
AcceptedCmp2            int64
Complain                int64
Z_CostContact           int64
Z_Revenue               int64
Response                int64
dtype: object
Tipo de dato despues de la conversion
 ID                              int64
Ye

In [27]:
# Paso 2 — duplicados
#se muestran las filas duplicadas
print("filas duplicadas:", df_practica.duplicated().sum())

#se eliminan las filas duplicadas y se reemplaza el dataframe
df_practica.drop_duplicates(inplace=True)
print("Filas duplicadas", df_practica.duplicated().sum())

filas duplicadas: 1
Filas duplicadas 0


In [29]:
# Paso 3 — valores faltantes
#se muestran valores faltantes del dataframe
print(df_practica.isnull().sum())

ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 1
Kidhome                0
Teenhome               0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Complain               0
Z_CostContact          0
Z_Revenue              0
Response               0
dtype: int64


In [32]:
# Paso 4 — exploración categórica
print(df_practica["Marital_Status"].unique())

print("\nLos valores no necestan normalizacion debido a que cumplen con el estandar de la columna")

['Together' 'Single' 'Married' 'Divorced' 'Widow']

Los valores no necestan normalizacion debido a que cumplen con el estandar de la columna


**Tu reporte de profiling:**

El data profiling o perfilado de datos es el proceso de examinar y resumir un conjunto de datos para poder comprender el contenido y la calidad del mismo. En esta practica se analizaron 2 datasets, uno de una base de datos web de netflix y el otro de marketing. Los proceos por el cual pasaron fueron la estandarizadion de columnas, eliminacion de duplicados y nulos, analisis de completitud de columnas y exploracion categorica.